In [1]:
# ============================================
# 4.1 Import Libraries
# ============================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================
# 4.2 Load Dataset
# ============================================

train_df = pd.read_csv("../data/train.csv")

print(train_df.shape)

(140700, 20)


In [3]:
# ============================================
# 4.3 Create Backup
# ============================================

train = train_df.copy()

train.drop(columns=["id", "Name"], inplace=True)

print("Backup Created")

Backup Created


In [4]:
# ============================================
# 4.4 Separate Columns
# ============================================

numerical_columns = train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns.remove("Depression")

print(numerical_columns)
print(categorical_columns)

['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Work/Study Hours', 'Financial Stress']
['Gender', 'City', 'Working Professional or Student', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']


In [5]:
# ============================================
# 4.5 Handle Missing Values
# ============================================

num_imputer = SimpleImputer(strategy="median")

train[numerical_columns] = num_imputer.fit_transform(
    train[numerical_columns]
)

cat_imputer = SimpleImputer(strategy="most_frequent")

train[categorical_columns] = cat_imputer.fit_transform(
    train[categorical_columns]
)

print("Missing Values Handled")

Missing Values Handled


In [6]:
# ============================================
# 4.6 Encode Categorical Columns
# ============================================

label_encoders = {}

for column in categorical_columns:

    train[column] = train[column].astype(str)

    le = LabelEncoder()

    train[column] = le.fit_transform(
        train[column]
    )

    label_encoders[column] = le

print("Encoding Completed")

Encoding Completed


In [7]:
# ============================================
# 4.7 Feature Target Split
# ============================================

X = train.drop("Depression", axis=1)

y = train["Depression"]

print(X.shape)
print(y.shape)

(140700, 17)
(140700,)


In [8]:
# ============================================
# 4.8 Train Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print(X_train.shape)

(112560, 17)


In [9]:
# ============================================
# 4.9 Feature Scaling
# ============================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

print("Scaling Completed")

Scaling Completed


In [10]:
# ============================================
# 4.10 Convert To Tensor
# ============================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1,1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1,1)

print("Tensor Conversion Completed")

Tensor Conversion Completed


In [11]:
# ============================================
# 4.11 Hyperparameter Tuning
# ============================================

results = []

In [12]:
# ============================================
# 4.12 Try Different Parameters
# ============================================

hidden_sizes = [
    [64, 32],
    [128, 64]
]

dropouts = [0.2, 0.3]

learning_rates = [0.001, 0.0005]

epochs_list = [50, 100]

In [13]:
# ============================================
# 4.13 Model Training Loop
# ============================================

for hidden in hidden_sizes:

    for dropout in dropouts:

        for lr in learning_rates:

            for epochs in epochs_list:

                class DepressionModel(nn.Module):

                    def __init__(self):

                        super().__init__()

                        self.network = nn.Sequential(

                            nn.Linear(17, hidden[0]),

                            nn.ReLU(),

                            nn.Dropout(dropout),

                            nn.Linear(hidden[0], hidden[1]),

                            nn.ReLU(),

                            nn.Dropout(dropout),

                            nn.Linear(hidden[1], 1)
                        )

                    def forward(self, x):

                        return self.network(x)

                model = DepressionModel()

                criterion = nn.BCEWithLogitsLoss()

                optimizer = optim.Adam(
                    model.parameters(),
                    lr=lr
                )

                # Training
                for epoch in range(epochs):

                    model.train()

                    outputs = model(X_train_tensor)

                    loss = criterion(
                        outputs,
                        y_train_tensor
                    )

                    optimizer.zero_grad()

                    loss.backward()

                    optimizer.step()

                # Evaluation
                model.eval()

                with torch.no_grad():

                    logits = model(X_test_tensor)

                    probs = torch.sigmoid(logits)

                    preds = (probs >= 0.5).float()

                y_pred = preds.numpy()

                accuracy = accuracy_score(
                    y_test,
                    y_pred
                )

                precision = precision_score(
                    y_test,
                    y_pred
                )

                recall = recall_score(
                    y_test,
                    y_pred
                )

                f1 = f1_score(
                    y_test,
                    y_pred
                )

                results.append({

                    "Hidden Layers": hidden,

                    "Dropout": dropout,

                    "Learning Rate": lr,

                    "Epochs": epochs,

                    "Accuracy": accuracy,

                    "Precision": precision,

                    "Recall": recall,

                    "F1 Score": f1
                })

                print(
                    hidden,
                    dropout,
                    lr,
                    epochs,
                    f1
                )

[64, 32] 0.2 0.001 50 0.3455245428296439
[64, 32] 0.2 0.001 100 0.8117079058539529
[64, 32] 0.2 0.0005 50 0.1913993655269651
[64, 32] 0.2 0.0005 100 0.5746410186941209
[64, 32] 0.3 0.001 50 0.4001847006310605
[64, 32] 0.3 0.001 100 0.7924723399855237
[64, 32] 0.3 0.0005 50 0.05585106382978723
[64, 32] 0.3 0.0005 100 0.658679645575939
[128, 64] 0.2 0.001 50 0.7804422298349424
[128, 64] 0.2 0.001 100 0.8209943659187506
[128, 64] 0.2 0.0005 50 0.5313466404273264
[128, 64] 0.2 0.0005 100 0.8025244299674267
[128, 64] 0.3 0.001 50 0.781256394516063
[128, 64] 0.3 0.001 100 0.8216766115621638
[128, 64] 0.3 0.0005 50 0.5208450704225353
[128, 64] 0.3 0.0005 100 0.7885535900104058


In [14]:
# ============================================
# 4.14 Results DataFrame
# ============================================

results_df = pd.DataFrame(results)

results_df

,Hidden Layers,Dropout,Learning Rate,Epochs,Accuracy,Precision,Recall,F1 Score
0,"[64, 32]",0.2,0.0010,50,0.855011,0.960749,0.210640,0.345525
1,"[64, 32]",0.2,0.0010,100,0.933475,0.835577,0.789165,0.811708
2,"[64, 32]",0.2,0.0005,50,0.836958,0.967914,0.106200,0.191399
3,"[64, 32]",0.2,0.0005,100,0.888415,0.934773,0.414825,0.574641
4,"[64, 32]",0.3,0.0010,50,0.861514,0.939306,0.254254,0.400185
5,"[64, 32]",0.3,0.0010,100,0.928678,0.840720,0.749462,0.792472
6,"[64, 32]",0.3,0.0005,50,0.823383,0.973510,0.028750,0.055851
7,"[64, 32]",0.3,0.0005,100,0.902807,0.910000,0.516135,0.658680
8,"[128, 64]",0.2,0.0010,50,0.924840,0.831637,0.735185,0.780442
9,"[128, 64]",0.2,0.0010,100,0.935643,0.829936,0.812243,0.820994


In [15]:
# ============================================
# 4.15 Best Model
# ============================================

best_model = results_df.sort_values(
    by="F1 Score",
    ascending=False
).head(1)

best_model

,Hidden Layers,Dropout,Learning Rate,Epochs,Accuracy,Precision,Recall,F1 Score
13,"[128, 64]",0.3,0.001,100,0.935217,0.821918,0.821436,0.821677


In [16]:
# ============================================
# 4.16 Final Message
# ============================================

print("Hyperparameter Tuning Completed")

Hyperparameter Tuning Completed
